# FitzHugh–Nagumo model (ODE) solved with time-marching PINNs (PyTorch)

Same problem as `FitzHugh-Nagumo-RK45.ipynb`:

$$
\frac{dv}{dt} = v - \frac{v^3}{3} - w + I_{\text{ext}}, \qquad
\frac{dw}{dt} = \frac{1}{\tau}\,\big(v + a - b\,w\big), \qquad
v(t_0)=v_0,\; w(t_0)=w_0 .
$$

The solution is approximated by physics-informed neural networks trained only on the ODE
residual (no solution data). The RK45 solution is used purely for validation.

### Why time-marching?

With these parameters the equilibrium $(v^*, w^*) \approx (-0.80, -0.13)$ is a *weakly* unstable
focus (eigenvalues $\approx 0.14 \pm 0.19i$). A single network on the whole interval $[0, 200]$
tends to converge to a trajectory that spirals into this equilibrium: it has a tiny residual but
misses every spike. Causal loss weighting delays that collapse but did not fix it in our tests
(relative error $\approx 0.7$ after long training).

Instead, $[t_0, t_{\text{final}}]$ is split into windows $[t_k, t_{k+1}]$. Window $k$ has its own
small MLP, and its initial condition is the end state of window $k-1$. Each window is short enough
that the optimiser reaches residuals $\sim 10^{-7}$, so the fake equilibrium solution can't take hold.

## Parameters (edit here)

In [ ]:
# ---------------- ODE parameters (same as the RK45 notebook) ----------------
a     = 0.7     # recovery offset
b     = 0.8     # recovery damping
tau   = 12.5    # time-scale separation (w is slower for larger tau)
I_ext = 0.5     # external stimulus current

# ---------------- Initial conditions ----------------
v0 = -1.0
w0 = 1.0

# ---------------- Time interval ----------------
t0      = 0.0
t_final = 200.0
n_eval  = 4000     # output grid for plots / comparison

# ---------------- Time-marching PINN settings ----------------
window_length  = 10.0   # length of each time window
n_colloc       = 400    # collocation points per window (uniform grid)
hidden_layers  = 3
hidden_width   = 32
adam_epochs    = 500    # Adam warm-up per window
adam_lr        = 1.0e-3
lbfgs_max_iter = 3000   # L-BFGS refinement per window
seed           = 0
num_threads    = 16     # CPU threads for torch

## Imports and setup

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from scipy.integrate import solve_ivp
from torch import nn

torch.manual_seed(seed)
np.random.seed(seed)
torch.set_num_threads(num_threads)

# First-order ODE, small networks: CPU + float64 is fast and lets L-BFGS
# drive the residual down to ~1e-7.
device = torch.device("cpu")
dtype = torch.float64
print(f"torch {torch.__version__}, device {device}, dtype {dtype}")

## Model

Inside window $[t_a, t_b]$, with $s = (t - t_a)/(t_b - t_a) \in [0, 1]$,

$$
\big(v, w\big)(t) = \big(v_a, w_a\big) + s\,\mathcal{N}_\theta(2s - 1),
$$

so the initial condition of each window (and continuity between windows) holds exactly.
The loss is the mean squared ODE residual on a uniform grid, with the $w$-equation
multiplied by $\tau$ so both residuals are $O(1)$:

$$
\mathcal{L} = \frac{1}{N}\sum_i \Big(\dot v - v + \tfrac{v^3}{3} + w - I_{\text{ext}}\Big)^2
+ \Big(\tau \dot w - v - a + b w\Big)^2 .
$$

In [ ]:
class WindowPINN(nn.Module):
    # MLP on one time window with a hard-coded initial condition.

    def __init__(self, t_a, t_b, y_a, hidden_layers, hidden_width):
        super().__init__()
        self.t_a, self.t_b = t_a, t_b
        layers, width_in = [], 1
        for _ in range(hidden_layers):
            linear = nn.Linear(width_in, hidden_width)
            nn.init.xavier_normal_(linear.weight)
            nn.init.zeros_(linear.bias)
            layers += [linear, nn.Tanh()]
            width_in = hidden_width
        output = nn.Linear(width_in, 2)
        nn.init.xavier_normal_(output.weight)
        nn.init.zeros_(output.bias)
        layers.append(output)
        self.net = nn.Sequential(*layers)
        self.register_buffer("y_a", torch.as_tensor(y_a, dtype=dtype))

    def forward(self, t):
        s = (t - self.t_a) / (self.t_b - self.t_a)
        return self.y_a + s * self.net(2.0 * s - 1.0)


def ode_residual(model, t):
    t = t.detach().requires_grad_(True)
    y = model(t)
    v, w = y[:, 0:1], y[:, 1:2]
    dv = torch.autograd.grad(v, t, torch.ones_like(v), create_graph=True)[0]
    dw = torch.autograd.grad(w, t, torch.ones_like(w), create_graph=True)[0]
    r_v = dv - (v - v**3 / 3.0 - w + I_ext)
    r_w = tau * dw - (v + a - b * w)
    return r_v, r_w


def residual_loss(model, t):
    r_v, r_w = ode_residual(model, t)
    return (r_v**2 + r_w**2).mean()

## Train window by window (Adam warm-up, then L-BFGS)

In [ ]:
def train_window(t_a, t_b, y_a):
    model = WindowPINN(t_a, t_b, y_a, hidden_layers, hidden_width).to(device, dtype)
    t_col = torch.linspace(t_a, t_b, n_colloc, dtype=dtype, device=device)[:, None]
    history = []

    adam = torch.optim.Adam(model.parameters(), lr=adam_lr)
    for _ in range(adam_epochs):
        adam.zero_grad()
        loss = residual_loss(model, t_col)
        loss.backward()
        adam.step()
        history.append(loss.item())

    lbfgs = torch.optim.LBFGS(
        model.parameters(),
        lr=1.0,
        max_iter=lbfgs_max_iter,
        history_size=100,
        tolerance_grad=1e-12,
        tolerance_change=1e-16,
        line_search_fn="strong_wolfe",
    )

    def closure():
        lbfgs.zero_grad()
        loss = residual_loss(model, t_col)
        loss.backward()
        history.append(loss.item())
        return loss

    lbfgs.step(closure)
    return model, history


edges = np.arange(t0, t_final + 1e-9, window_length)
if edges[-1] < t_final:
    edges = np.append(edges, t_final)

models, histories = [], []
y_a = [v0, w0]
start = time.time()
for t_a, t_b in zip(edges[:-1], edges[1:]):
    model, history = train_window(float(t_a), float(t_b), y_a)
    with torch.no_grad():
        y_a = model(torch.tensor([[t_b]], dtype=dtype, device=device))[0].tolist()
    models.append(model)
    histories.append(history)
    print(f"window [{t_a:6.1f}, {t_b:6.1f}]  final loss {history[-1]:.2e}  "
          f"({len(history)} evals, {time.time() - start:.0f}s)")

## Evaluate the PINN and the RK45 reference on the output grid

In [ ]:
def pinn_predict(t):
    # Evaluate the piecewise PINN at times t (1D array).
    t = np.asarray(t, dtype=np.float64)
    idx = np.clip(np.searchsorted(edges, t, side="right") - 1, 0, len(models) - 1)
    out = np.empty((t.size, 2))
    with torch.no_grad():
        for k, model in enumerate(models):
            mask = idx == k
            if mask.any():
                tk = torch.tensor(t[mask][:, None], dtype=dtype, device=device)
                out[mask] = model(tk).cpu().numpy()
    return out[:, 0], out[:, 1]


def fitzhugh_nagumo(t, y, a, b, tau, I_ext):
    v, w = y
    dv = v - v**3 / 3.0 - w + I_ext
    dw = (v + a - b * w) / tau
    return [dv, dw]


t = np.linspace(t0, t_final, n_eval)
v, w = pinn_predict(t)

# Tight tolerances so the reference error is negligible next to the PINN error.
sol = solve_ivp(fitzhugh_nagumo, (t0, t_final), [v0, w0], method="RK45", t_eval=t,
                args=(a, b, tau, I_ext), rtol=1e-10, atol=1e-12)
v_ref, w_ref = sol.y

err_v, err_w = np.abs(v - v_ref), np.abs(w - w_ref)
rel_l2 = np.sqrt(np.sum((v - v_ref)**2 + (w - w_ref)**2) / np.sum(v_ref**2 + w_ref**2))
print(f"relative L2 error (v, w): {rel_l2:.3e}")
print(f"max |v - v_ref| = {err_v.max():.3e},  max |w - w_ref| = {err_w.max():.3e}")

## Time series: PINN vs RK45

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
for ax, pinn, ref, name in [(axes[0], v, v_ref, "v"), (axes[1], w, w_ref, "w")]:
    ax.plot(t, ref, color="lightgray", lw=4, label=f"{name} RK45")
    ax.plot(t, pinn, "--", lw=1.5, label=f"{name} PINN")
    for e in edges[1:-1]:
        ax.axvline(e, color="k", lw=0.3, alpha=0.3)
    ax.set_ylabel(name)
    ax.legend(loc="upper right")
    ax.grid(alpha=0.3)
axes[1].set_xlabel("t")
axes[0].set_title(f"FitzHugh–Nagumo, time-marching PINN  (a={a}, b={b}, tau={tau}, I_ext={I_ext})")
plt.tight_layout()
plt.show()

## Pointwise error

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.semilogy(t, err_v + 1e-16, label="|v - v_RK45|")
ax.semilogy(t, err_w + 1e-16, label="|w - w_RK45|")
ax.set_xlabel("t")
ax.set_ylabel("absolute error")
ax.set_title("PINN error (grows slowly as window end-states are passed forward)")
ax.legend()
ax.grid(alpha=0.3, which="both")
plt.tight_layout()
plt.show()

## Phase plane

In [ ]:
vv = np.linspace(-2.7, 2.7, 400)
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(vv, vv - vv**3 / 3 + I_ext, "--", label=r"$\dot v = 0$")
ax.plot(vv, (vv + a) / b, "--", label=r"$\dot w = 0$")
ax.plot(v_ref, w_ref, color="lightgray", lw=4, label="RK45")
ax.plot(v, w, "k", lw=1.0, label="PINN")
ax.plot(v0, w0, "o", color="tab:red", label="initial condition")
ax.set_xlim(-2.7, 2.7)
ax.set_ylim(-1.2, 2.2)
ax.set_xlabel("v")
ax.set_ylabel("w")
ax.set_title("Phase plane")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

## Training history per window

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
cmap = plt.get_cmap("viridis")
for k, history in enumerate(histories):
    ax.semilogy(history, color=cmap(k / max(len(histories) - 1, 1)), lw=0.8)
ax.axvline(adam_epochs, color="k", ls=":", lw=1, label="Adam → L-BFGS")
ax.set_xlabel("iteration (Adam steps, then L-BFGS function evaluations)")
ax.set_ylabel("residual loss")
ax.set_title("Loss history (colour = window index, early → late)")
ax.legend()
ax.grid(alpha=0.3, which="both")
plt.tight_layout()
plt.show()